# Post-processing

> **Install for this tutorial:** `pip install "scope-profiler[pproc]"`;
> pandas is used for the DataFrame examples.

This notebook is a tour of the analysis API: `ProfilingResults` and the
`MPIRegion` / `Region` objects it hands out.

The one thing worth memorising: **every duration and timestamp in this API is in
seconds**, converted from the nanoseconds stored on disk.


In [ ]:
import os
import tempfile

# Work in a scratch directory so the tutorial leaves no files behind.
os.chdir(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))

In [ ]:
import time

from scope_profiler import ProfileManager, read_h5

ProfileManager.setup(file_path="profiling_data.h5", label="tutorial 2")


@ProfileManager.profile("assemble")
def assemble(scale):
    time.sleep(0.004 * scale)


with ProfileManager.region("setup"):
    time.sleep(0.02)

for step in range(5):
    with ProfileManager.region("timestep"):
        assemble(scale=1 + step % 3)
        with ProfileManager.region("solve"):
            time.sleep(0.003)

ProfileManager.finalize(verbose=False)
results = read_h5("profiling_data.h5")
results

## `ProfilingResults` is a mapping of regions

It behaves like an ordered mapping from region name to region, so the
usual Python idioms work.

In [ ]:
print("regions:", results.region_names)
print("count:", len(results))
print("'solve' recorded?", "solve" in results)
print("ranks in file:", results.num_ranks)

for region in results:
    print(f"  {region.name:<10} {region.num_calls:>3} calls")

Asking for something that is not there tells you what *is* there:

In [ ]:
try:
    results["sovle"]  # typo
except KeyError as exc:
    print(exc)

## Two levels: MPIRegion and Region

`results["solve"]` returns an **`MPIRegion`** — the region across every rank that
recorded it. Indexing it by rank gives a **`Region`**, the timings from that one
rank.

For a serial run there is only rank 0, but the two levels are the same API you
use for an MPI run.

In [ ]:
timestep = results["timestep"]

print("aggregated over ranks")
print("  ranks       :", timestep.ranks)
print("  num_calls   :", timestep.num_calls)
print("  total   [s] :", timestep.total_duration)
print("  average [s] :", timestep.average_duration)
print("  min/max [s] :", timestep.min_duration, timestep.max_duration)
print("  std     [s] :", timestep.std_duration)

rank0 = timestep[0]
print("\nrank 0 only")
print("  durations [s]:", rank0.durations)
# Timestamps are on a monotonic clock with an arbitrary origin; measure them
# from the start of the run, as events() does (see tutorial 5).
print("  start times [s]:", (rank0.start_times - results.time_origin).round(3))
print("  summary:", rank0.get_summary())

Per-rank breakdowns are available as dictionaries keyed by rank, which is where
load imbalance shows up in an MPI run:

In [ ]:
print("calls per rank :", timestep.num_calls_per_rank())
print("total per rank :", timestep.total_durations())
print("avg per rank   :", timestep.average_durations())
print("max per rank   :", timestep.max_durations())

## Summaries and DataFrames

`summary()` returns one dict per region, aggregated over ranks.
`print_summary()` formats the same data as a table — nested regions are indented
under their parent, with an `(own)` row for the time a parent spends outside its
children. It takes `sort=`, `columns=` and the usual `include` / `exclude`
filters; `suppress_notes=True` drops the footer of follow-up commands.

In [ ]:
results.print_summary(suppress_notes=True)

Each summary row carries the totals, the per-call statistics (`average`, `min`,
`max`, `std`, and the percentiles `p50`, `p95`, `p99`), the *exclusive* time that
excludes nested regions, and the load imbalance across ranks. Here are a few of
the fields:

In [ ]:
fields = (
    "name",
    "num_calls",
    "total_duration",
    "exclusive_duration",
    "p95",
    "imbalance",
)
for row in results.summary():
    print({key: row[key] for key in fields})

With pandas installed (it comes with the `pproc` extra), `to_dataframe()` gives
you the same data ready for sorting, filtering and plotting.

In [ ]:
frame = results.to_dataframe()
columns = ["name", "num_calls", "total_duration", "exclusive_duration", "p95_duration"]
frame[columns].sort_values("total_duration", ascending=False)

`per_rank=True` emits one row per (region, rank) instead — the shape you want
for load-balance analysis.

In [ ]:
per_rank = results.to_dataframe(per_rank=True)
per_rank[["name", "rank", "num_calls", "total_duration", "average_duration"]]

In [ ]:
# Share of total recorded time per region.
frame = frame.assign(share=lambda df: df.total_duration / df.total_duration.sum())
frame[["name", "num_calls", "total_duration", "share"]].sort_values(
    "share", ascending=False
)

## Selecting regions

`get_regions()`, `summary()`, `to_dataframe()` and the plotting functions all
take `include` / `exclude`, which are **regular expressions** matched against
the region name (via `re.match`, so they anchor at the start).

In [ ]:
print([region.name for region in results.get_regions(include="s")])
print([region.name for region in results.get_regions(exclude=["setup", "assemble"])])
print([row["name"] for row in results.summary(include=["solve", "timestep"])])

## Run metadata

Each file carries a description of the environment it was recorded in, which is
what tells two otherwise identical runs apart months later. By default that is a
*minimal* record: versions, platform, CPU model, rank and thread counts, the
start time and the run's `label`.

In [ ]:
for key, value in sorted(results.metadata.items()):
    print(f"{key:>24}: {value}")

`setup(metadata_detail="full")` adds what is more personal or more bulky: the
user and host name, the working directory, loaded environment modules, selected
environment variables such as `PATH`, and any `SLURM_*` job variables. It is off
by default so a profile can be shared without leaking that. Capturing the region
source text is a separate switch, `capture_region_source=True`.

In [ ]:
ProfileManager.setup(file_path="full.h5", metadata_detail="full", label="full metadata")
with ProfileManager.region("work"):
    pass
full = ProfileManager.finalize(verbose=False, return_results=True)

print("extra fields:", sorted(set(full.metadata) - set(results.metadata)))

`scope-profiler inspect` prints the metadata grouped and readable — together
with per-region statistics — straight from the command line. Long values such
as `PATH` are clipped unless you pass `--full`:

```bash
scope-profiler inspect profiling_data.h5
scope-profiler inspect profiling_data.h5 --metadata-only
scope-profiler inspect profiling_data.h5 --regions-only --sort total
```

`write_metadata_json()` exports the metadata instead, with one entry per file and
no clipping of long values:

In [ ]:
from scope_profiler.inspection import write_metadata_json

payload = write_metadata_json("profiling_data.h5", "metadata.json")
print(sorted(payload["files"][0]["metadata"])[:8])

## Exporting statistics

`collect_region_statistics()` builds a JSON-ready dict (durations in seconds,
with a `units` field spelling that out), and `write_region_statistics_json()`
writes it to a file. Both accept several runs at once for run comparisons.

In [ ]:
from scope_profiler import collect_region_statistics, write_region_statistics_json

stats = collect_region_statistics(results, include="solve")
print(stats["units"])
print(stats["files"][0]["region_statistics"]["solve"])

write_region_statistics_json(results, "region_statistics.json")
print("wrote region_statistics.json")

## Next

[3. Visualizing results](/scope-profiler/guide/03-visualization/) turns these numbers into Gantt,
flame and duration charts,
[5. Custom analysis](/scope-profiler/guide/05-custom-analysis/) goes below the summaries to the
individual calls, and
[8. Reports and comparison](/scope-profiler/guide/08-reports-and-comparison/) compares whole runs.